# 🚀 AsteroMesh — Training Pipeline
**Autonomous 3-D Asteroid Shape Reconstruction from Multimodal Partial Observations**

Team: **Poornima ki raat** (Agastya, Agam)

This notebook:
1. Clones the repo & installs deps
2. Generates synthetic training data from test shapes
3. Trains the dual-stream model
4. Visualises results

**⚠️ Enable GPU:** Runtime → Change runtime type → T4 GPU

## 1. Setup

In [ ]:
# Clone repo
!git clone https://github.com/Agamjot0123/Poornima_ki_raat.git
%cd Poornima_ki_raat

# Install dependencies
!pip install -q trimesh open3d pyshtools pyvista scipy numpy matplotlib tqdm pyyaml

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Generate Synthetic Training Data

We create simple asteroid-like shapes (sphere, ellipsoid, peanut) and generate synthetic observations.

In [ ]:
import numpy as np
import trimesh
import os
from pathlib import Path

# Create directories
os.makedirs('data/synthetic/train', exist_ok=True)
os.makedirs('data/synthetic/val', exist_ok=True)
os.makedirs('data/ground_truth', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('outputs/meshes', exist_ok=True)

def create_sphere(subdivisions=3):
    return trimesh.creation.icosphere(subdivisions=subdivisions)

def create_ellipsoid(a=1.0, b=0.7, c=0.5, subdivisions=3):
    mesh = trimesh.creation.icosphere(subdivisions=subdivisions)
    mesh.vertices *= np.array([a, b, c])
    return mesh

def create_peanut(subdivisions=3):
    """Create a peanut/bilobed shape like asteroid Itokawa."""
    mesh = trimesh.creation.icosphere(subdivisions=subdivisions)
    verts = mesh.vertices.copy()
    x, y, z = verts[:, 0], verts[:, 1], verts[:, 2]
    # Pinch in the middle along x-axis
    scale = 0.6 + 0.4 * np.abs(x)
    verts[:, 1] *= scale
    verts[:, 2] *= scale
    verts[:, 0] *= 1.5  # Elongate
    mesh.vertices = verts
    return mesh

def create_lumpy(subdivisions=3, seed=42):
    """Create a lumpy asteroid shape with random bumps."""
    np.random.seed(seed)
    mesh = trimesh.creation.icosphere(subdivisions=subdivisions)
    verts = mesh.vertices.copy()
    # Add random radial perturbations
    r = np.linalg.norm(verts, axis=1, keepdims=True)
    noise = np.random.randn(len(verts), 1) * 0.15
    verts = verts * (1 + noise)
    # Slight elongation
    verts *= np.array([1.2, 0.9, 0.8])
    mesh.vertices = verts
    return mesh

# Generate base shapes
shapes = {
    'sphere': create_sphere(),
    'ellipsoid_1': create_ellipsoid(1.0, 0.7, 0.5),
    'ellipsoid_2': create_ellipsoid(1.2, 0.6, 0.8),
    'ellipsoid_3': create_ellipsoid(0.8, 1.0, 0.6),
    'peanut': create_peanut(),
    'lumpy_1': create_lumpy(seed=42),
    'lumpy_2': create_lumpy(seed=123),
    'lumpy_3': create_lumpy(seed=456),
}

# Save ground truth meshes
for name, mesh in shapes.items():
    mesh.export(f'data/ground_truth/{name}.obj')

print(f'Created {len(shapes)} ground-truth shapes')
for name, mesh in shapes.items():
    print(f'  {name}: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

In [ ]:
# Generate synthetic observations from each shape
import torch
import sys
sys.path.append('/content/Poornima_ki_raat')

from src.data.synthetic_generator import (
    centre_and_normalise_mesh, compute_spharm_coefficients
)
from src.data.light_curve_loader import generate_synthetic_light_curve
from src.data.radar_loader import generate_synthetic_radar

MAX_DEGREE = 15  # Lower degree for faster training
NUM_COEFFS = 2 * (MAX_DEGREE + 1) ** 2
SAMPLES_PER_SHAPE = 80  # augmented samples per shape

print(f'SPHARM degree: {MAX_DEGREE}, coefficients: {NUM_COEFFS}')
print(f'Generating {SAMPLES_PER_SHAPE} samples per shape...')

sample_idx = 0

for name, mesh in shapes.items():
    print(f'Processing {name}...')
    mesh_copy = mesh.copy()
    mesh_copy = centre_and_normalise_mesh(mesh_copy)
    vertices = np.array(mesh_copy.vertices)
    faces = np.array(mesh_copy.faces)

    # Compute target SPHARM coefficients
    coeffs = compute_spharm_coefficients(mesh_copy, max_degree=MAX_DEGREE)

    for i in range(SAMPLES_PER_SHAPE):
        # Random viewing geometry
        view_theta = np.random.uniform(0, np.pi)
        view_phi = np.random.uniform(0, 2 * np.pi)
        view_dir = np.array([
            np.sin(view_theta) * np.cos(view_phi),
            np.sin(view_theta) * np.sin(view_phi),
            np.cos(view_theta)
        ])

        # Generate light curve
        lc = generate_synthetic_light_curve(
            vertices, faces, num_phases=512, viewing_direction=view_dir
        )

        # Generate radar image
        radar = generate_synthetic_radar(
            vertices, faces, image_size=224,
            viewing_angle=np.random.uniform(0, 2*np.pi),
            subradar_lat=np.random.uniform(-np.pi/4, np.pi/4)
        )

        # Add noise
        lc = lc + torch.randn_like(lc) * 0.05
        radar = torch.clamp(radar + torch.randn_like(radar) * 0.03, 0, 1)

        sample = {
            'light_curve': lc,
            'radar_image': radar,
            'spharm_coefficients': torch.tensor(coeffs, dtype=torch.float32),
        }

        # 90% train, 10% val
        split = 'train' if i < int(SAMPLES_PER_SHAPE * 0.9) else 'val'
        torch.save(sample, f'data/synthetic/{split}/sample_{sample_idx:06d}.pt')
        sample_idx += 1

train_count = len(list(Path('data/synthetic/train').glob('*.pt')))
val_count = len(list(Path('data/synthetic/val').glob('*.pt')))
print(f'\nDone! Train: {train_count}, Val: {val_count}, Total: {sample_idx}')

## 3. Train the Model

In [ ]:
import yaml
from src.models.asteromesh import AsteroMesh
from src.data.dataset import AsteroidDataset
from src.training.losses import CompositeLoss
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import time

# Config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 50
BATCH_SIZE = 8
LR = 0.001
MAX_DEGREE = 15
NUM_COEFFS = 2 * (MAX_DEGREE + 1) ** 2

print(f'Device: {DEVICE}')
print(f'SPHARM coefficients: {NUM_COEFFS}')

# Model
model = AsteroMesh(
    light_curve_length=512,
    radar_image_size=224,
    encoder_dim=256,
    fused_dim=512,
    max_degree=MAX_DEGREE,
    mesh_resolution=50,
    pretrained_backbone=True,
).to(DEVICE)

params = model.count_parameters()
print(f'\nParameters:')
for k, v in params.items():
    print(f'  {k}: {v:,}')

In [ ]:
# Datasets & loaders
train_ds = AsteroidDataset(
    'data/synthetic/train', mode='both',
    num_coefficients=NUM_COEFFS, augment=True
)
val_ds = AsteroidDataset(
    'data/synthetic/val', mode='both',
    num_coefficients=NUM_COEFFS, augment=False
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val: {len(val_ds)} samples, {len(val_loader)} batches')

# Loss, optimiser, scheduler
criterion = CompositeLoss(
    mse_weight=1.0, spharm_reg_weight=0.01, max_degree=MAX_DEGREE
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')

for epoch in range(EPOCHS):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_total = 0
    for batch in train_loader:
        lc = batch['light_curve'].to(DEVICE)
        radar = batch['radar_image'].to(DEVICE)
        target = batch['spharm_coefficients'].to(DEVICE)

        pred = model(lc, radar)
        losses = criterion(pred, target)

        optimizer.zero_grad()
        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_total += losses['total'].item()

    train_avg = train_total / len(train_loader)

    # --- Validate ---
    model.eval()
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            lc = batch['light_curve'].to(DEVICE)
            radar = batch['radar_image'].to(DEVICE)
            target = batch['spharm_coefficients'].to(DEVICE)
            pred = model(lc, radar)
            losses = criterion(pred, target)
            val_total += losses['total'].item()

    val_avg = val_total / max(len(val_loader), 1)

    scheduler.step()
    history['train_loss'].append(train_avg)
    history['val_loss'].append(val_avg)

    # Save best
    is_best = val_avg < best_val
    if is_best:
        best_val = val_avg
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'optimizer_state_dict': optimizer.state_dict(),
                     'loss': val_avg}, 'checkpoints/best_model.pth')

    elapsed = time.time() - t0
    star = ' ★' if is_best else ''
    if (epoch+1) % 5 == 0 or epoch == 0 or is_best:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_avg:.6f} | Val: {val_avg:.6f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}s{star}')

print(f'\nBest val loss: {best_val:.6f}')

## 4. Results & Visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(history['train_loss'], label='Train', linewidth=2)
ax.plot(history['val_loss'], label='Val', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training History')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Load best model and reconstruct test shapes
checkpoint = torch.load('checkpoints/best_model.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]}')

# Reconstruct from a validation sample
val_sample = val_ds[0]
lc = val_sample['light_curve'].unsqueeze(0).to(DEVICE)
radar = val_sample['radar_image'].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred_coeffs = model(lc, radar)

print(f'Predicted coefficients shape: {pred_coeffs.shape}')

# Decode to mesh
pred_mesh = model.mesh_decoder.to_mesh(pred_coeffs[0].cpu())
print(f'Reconstructed mesh: {len(pred_mesh.vertices)} vertices, {len(pred_mesh.faces)} faces')
print(f'Watertight: {pred_mesh.is_watertight}')

# Save
pred_mesh.export('outputs/meshes/reconstruction.obj')
print('Saved to outputs/meshes/reconstruction.obj')

In [ ]:
# Visualise reconstruction vs a ground truth
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

gt_mesh = trimesh.load('data/ground_truth/ellipsoid_1.obj', force='mesh')

fig = plt.figure(figsize=(14, 6))

for idx, (mesh, title, color) in enumerate([
    (pred_mesh, 'Predicted', '#4CAF50'),
    (gt_mesh, 'Ground Truth (ellipsoid)', '#2196F3')
]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)
    poly = Poly3DCollection(verts[faces], alpha=0.6, facecolor=color,
                             edgecolor='gray', linewidth=0.1)
    ax.add_collection3d(poly)
    lim = np.max(np.abs(verts)) * 1.2
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

plt.suptitle('AsteroMesh Reconstruction', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compute evaluation metrics
from src.evaluation.metrics import compute_all_metrics, sample_points_from_mesh

pred_pts = sample_points_from_mesh(pred_mesh, 5000)
gt_pts = sample_points_from_mesh(gt_mesh, 5000)

metrics = compute_all_metrics(pred_pts, gt_pts)

print('\n=== Evaluation Metrics ===')
print(f'  Hausdorff Distance: {metrics["hausdorff_distance"]:.6f}  (↓)')
print(f'  Chamfer Distance:   {metrics["chamfer_distance"]:.6f}  (↓)')
print(f'  RMSE:               {metrics["rmse"]:.6f}  (↓)')
print(f'  Volumetric IoU:     {metrics["volumetric_iou"]:.4f}  (↑)')
print(f'  Completeness:       {metrics["completeness"]:.2f}%  (↑)')

## 5. Save to Drive (Optional)
Uncomment below to save the trained model to Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp checkpoints/best_model.pth /content/drive/MyDrive/asteromesh_best.pth
# !cp outputs/meshes/reconstruction.obj /content/drive/MyDrive/reconstruction.obj
# print('Saved to Drive!')